In [70]:
import polars as pl
import numpy as np
import string


In [71]:
filtered = pl.scan_parquet("../ddl_judicial_data/ddl_judicial_data_2018_processed/cases_labeled.parquet")

In [72]:
filtered.lazy().collect_schema()

Schema([('ddl_case_id', String),
        ('year', Int64),
        ('state_code', Int64),
        ('dist_code', Int64),
        ('court_no', Int64),
        ('cino', String),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('type_name', Int64),
        ('purpose_name', Int64),
        ('disp_name', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('date_first_list', Date),
        ('date_last_list', Date),
        ('date_next_list', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('ddl_filing_judge_id', Int64),
        ('ddl_decision_judge_id', Int64),
        ('female_judge_filing', String),
        ('start_date_filing', Date),
        ('end_da

In [73]:
filtered = filtered.filter(pl.col("criminal") == 1)

filtered_ids = filtered.select("ddl_case_id").collect()

sampled_ids = filtered_ids.sample(n=50_000, seed=42)

cases_sampled = filtered.join(sampled_ids.lazy(), on="ddl_case_id", how="semi").collect()

In [ ]:
cases_sampled.write_parquet("../data_sampled/cases_sampled.parquet")

In [75]:
cases_shuffled = cases_sampled.sample(fraction = 1, shuffle = True, seed = 38)

In [76]:
cases_shuffled_indexed = cases_shuffled.with_row_index(name = "record_id")
cases_answer_key = cases_shuffled_indexed.select(["record_id", "ddl_case_id"])

In [77]:
print(cases_shuffled.shape)

corrupted_cases = cases_shuffled_indexed.drop("ddl_case_id", "cino", "ddl_filing_judge_id", "ddl_decision_judge_id")

corrupted_cases.shape
 

(50000, 39)


(50000, 36)

In [78]:
random_generated = np.random.default_rng(seed = 50)
which_rows = random_generated.random(50_000) < 0.15

how_many_days = random_generated.integers(1, 4, 50_000) * random_generated.choice([-1, 1], 50_000)

corrupted_cases = corrupted_cases.with_columns([pl.Series(values=which_rows, name="shift_flag"), pl.Series(values=how_many_days, name="shift_days")])

corrupted_cases = corrupted_cases.with_columns(pl.when(pl.col("shift_flag")).then(pl.col("date_of_filing") + pl.duration(days=pl.col("shift_days"))).otherwise(pl.col("date_of_filing")).alias("date_of_filing"))

In [79]:
which_rows2 = random_generated.random(50_000) < 0.15

how_many_days2 = random_generated.integers(1, 4, 50_000) * random_generated.choice([-1, 1], 50_000)

corrupted_cases = corrupted_cases.with_columns([pl.Series(values=which_rows2, name="shift_flag2"), pl.Series(values=how_many_days2, name="shift_days2")])

corrupted_cases = corrupted_cases.with_columns(pl.when(pl.col("shift_flag2")).then(pl.col("date_of_decision") + pl.duration(days=pl.col("shift_days2"))).otherwise(pl.col("date_of_decision")).alias("date_of_decision"))

In [80]:
corrupted_cases = corrupted_cases.drop(["shift_flag", "shift_flag2", "shift_days", "shift_days2", "date_first_list", "date_last_list", "date_next_list"])

In [81]:
def add_typo(name):
    if name is None or len(name) < 2:
        return name
    i = random_generated.integers(0, len(name) - 1) 
    op = random_generated.choice(["delete", "swap", "replace"])
    if op == "delete": 
        return name[:i] + name[i+1:]
    elif op == "swap":
        return name[:i] + name[i+1] + name[i] + name[i+2:]
    else:
        return name[:i] + random_generated.choice(list(string.ascii_lowercase)) + name[i+1:]

In [82]:
add_typo("India")

'gndia'

In [83]:
typo_flag_district = random_generated.random(50000) < 0.1
corrupted_cases = corrupted_cases.with_columns(pl.Series("typo_flag_district", typo_flag_district)).with_columns(pl.when(pl.col("typo_flag_district")).then(pl.col("district_name").map_elements(add_typo, return_dtype=pl.String)).otherwise(pl.col("district_name")).alias("district_name"))

In [84]:
typo_flag_court = random_generated.random(50000) < 0.1
corrupted_cases = corrupted_cases.with_columns(pl.Series("typo_flag_court", typo_flag_court)).with_columns(pl.when(pl.col("typo_flag_court")).then(pl.col("court_name").map_elements(add_typo, return_dtype=pl.String)).otherwise(pl.col("court_name")).alias("court_name"))

In [85]:
corrupted_cases = corrupted_cases.drop(["typo_flag_court", "typo_flag_district"])

In [86]:
corrupted_cases.height

50000

In [87]:
blank_flag_name_type = random_generated.random(50_000) < 0.05
blank_flag_female_d = random_generated.random(50_000) < 0.05
blank_flag_female_p = random_generated.random(50_000) < 0.05

In [88]:
corrupted_cases = corrupted_cases.with_columns(pl.Series("blank_flag_name_type", blank_flag_name_type), pl.Series("blank_flag_female_d", blank_flag_female_d), pl.Series("blank_flag_female_p", blank_flag_female_p)).with_columns(pl.when("blank_flag_name_type").then(pl.lit(None, dtype=pl.String)).otherwise("type_name_s").alias("type_name_s"), pl.when("blank_flag_female_d").then(pl.lit(None, dtype=pl.String)).otherwise("female_defendant").alias("female_defendant"), pl.when("blank_flag_female_p").then(pl.lit(None, dtype=pl.String)).otherwise("female_petitioner").alias("female_petitioner"))

In [89]:
corrupted_cases = corrupted_cases.drop(["blank_flag_female_d", "blank_flag_female_p", "blank_flag_name_type"])

In [90]:
corrupted_cases.lazy().collect_schema()

Schema([('record_id', UInt32),
        ('year', Int64),
        ('state_code', Int64),
        ('dist_code', Int64),
        ('court_no', Int64),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('type_name', Int64),
        ('purpose_name', Int64),
        ('disp_name', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('female_judge_filing', String),
        ('start_date_filing', Date),
        ('end_date_filing', Date),
        ('female_judge_decision', String),
        ('start_date_decision', Date),
        ('end_date_decision', Date),
        ('act_s', List(String)),
        ('section_s', List(String)),
     

In [91]:
corrupted_cases = corrupted_cases.drop("state_code", "dist_code", "court_no", "type_name", "purpose_name", "disp_name", "start_date_filing", "end_date_filing", "start_date_decision", "end_date_decision")

In [92]:
corrupted_cases.write_parquet("../data_sampled/corrupted_cases_labeled.parquet")
cases_answer_key.write_parquet("../data_sampled/cases_answer_key.parquet")